# Analisis Financiero:
- Clasificación de categorias
- Clasificación de perfil financiero
- Recomendaciones Automaticas

In [1]:
!pip install -q scikit-learn==1.9.0 joblib==1.5.3
#Python 3.12.2


[notice] A new release of pip is available: 24.0 -> 26.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
# Librerias
import pandas as pd
import numpy as np
import os
import joblib


## Importar y conocer la base de datos

In [3]:
# LLamando dataset
df1 = pd.read_csv("datasets/personal_finance_1.csv")
df1.head(10)

,Date,Transaction Description,Category,Amount,Type
0,2020-01-02,Score each.,Food & Drink,1485.69,Expense
1,2020-01-02,Quality throughout.,Utilities,1475.58,Expense
2,2020-01-04,Instead ahead despite measure ago.,Rent,1185.08,Expense
3,2020-01-05,Information last everything thank serve.,Investment,2291.00,Income
4,2020-01-13,Future choice whatever from.,Food & Drink,1126.88,Expense
5,2020-01-14,Benefit suggest page southern.,Shopping,448.68,Expense
6,2020-01-18,By two bad fall pick.,Food & Drink,1520.03,Expense
7,2020-01-19,Court attorney product significant world.,Other,3287.00,Income
8,2020-01-21,Herself law.,Entertainment,1914.85,Expense
9,2020-01-25,Have decide environment.,Rent,766.06,Expense


In [4]:
df2 = pd.read_csv("datasets/personal_finance_2.csv")
df2.head(10)

,date,user_id,monthly_income,monthly_expense_total,savings_rate,budget_goal,financial_scenario,credit_score,debt_to_income_ratio,loan_payment,...,discretionary_spending,essential_spending,income_type,rent_or_mortgage,category,cash_flow_status,financial_advice_score,financial_stress_level,actual_savings,savings_goal_met
0,2019-01-01,1584,3119.58,3212.07,0.38,3676.11,inflation,721.0,0.56,125.77,...,857.55,1910.85,Freelance,1501.65,Investments,Positive,8.3,Low,0.00,0
1,2019-01-31,1045,3262.44,3732.81,0.10,2607.17,inflation,670.0,0.42,454.19,...,534.51,3165.20,Salary,1603.17,Investments,Positive,22.6,Low,0.00,0
2,2019-03-02,1756,2931.20,3335.58,0.15,3004.14,inflation,691.0,0.24,971.82,...,353.67,1504.56,Freelance,1097.82,Healthcare,Positive,58.8,Low,0.00,0
3,2019-04-01,1724,3506.79,2327.59,0.17,3346.97,normal,717.0,0.16,482.76,...,594.08,1450.72,Freelance,1155.64,Groceries,Positive,74.5,Low,1179.20,0
4,2019-05-01,1600,4606.87,2182.58,0.34,2670.09,inflation,795.0,0.25,263.74,...,556.86,1000.00,Salary,1170.86,Utilities,Negative,38.7,High,2424.29,0
5,2019-05-31,1985,4028.39,3876.57,0.26,2079.72,recession,643.0,0.43,323.20,...,431.04,2330.01,Salary,548.67,Healthcare,Positive,53.7,Medium,151.82,0
6,2019-06-30,1194,4145.45,3035.61,0.15,2215.80,normal,683.0,0.30,609.88,...,443.50,2860.30,Freelance,637.04,Transportation,Positive,74.7,High,1109.84,0
7,2019-07-30,1600,3202.37,2689.51,0.25,3650.48,normal,655.0,0.24,524.64,...,602.88,3526.55,Salary,1453.18,Entertainment,Neutral,83.1,Medium,512.86,0
8,2019-08-29,1835,4765.11,2628.56,0.06,2813.81,normal,715.0,0.52,103.30,...,733.67,2503.65,Salary,1675.75,Education,Negative,71.6,Low,2136.55,0
9,2019-09-28,1454,3581.79,3370.80,0.13,2195.92,normal,749.0,0.52,643.48,...,464.34,1691.26,Salary,682.16,Entertainment,Positive,77.9,High,210.99,0


In [5]:
df2.info()

<class 'pandas.DataFrame'>
RangeIndex: 3000 entries, 0 to 2999
Data columns (total 25 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   date                    3000 non-null   str    
 1   user_id                 3000 non-null   int64  
 2   monthly_income          3000 non-null   float64
 3   monthly_expense_total   3000 non-null   float64
 4   savings_rate            3000 non-null   float64
 5   budget_goal             3000 non-null   float64
 6   financial_scenario      3000 non-null   str    
 7   credit_score            3000 non-null   float64
 8   debt_to_income_ratio    3000 non-null   float64
 9   loan_payment            3000 non-null   float64
 10  investment_amount       3000 non-null   float64
 11  subscription_services   3000 non-null   int64  
 12  emergency_fund          3000 non-null   float64
 13  transaction_count       3000 non-null   int64  
 14  fraud_flag              3000 non-null   int64  
 15

In [6]:
#Renombrar Columnas
traduccion = {
    'date': 'fecha',
    'user_id': 'id_usuario',
    'monthly_income': 'ingreso_mensual',
    'monthly_expense_total': 'gasto_mensual_total',
    'savings_rate': 'tasa_ahorro',
    'budget_goal': 'objetivo_presupuesto',
    'financial_scenario': 'escenario_financiero',
    'credit_score': 'puntaje_crediticio',
    'debt_to_income_ratio': 'relacion_deuda_ingreso',
    'loan_payment': 'pago_prestamo',
    'investment_amount': 'monto_inversion',
    'subscription_services': 'servicios_suscripcion',
    'emergency_fund': 'fondo_emergencia',
    'transaction_count': 'cantidad_transacciones',
    'fraud_flag': 'indicador_fraude',
    'discretionary_spending': 'gastos_discrecionales',
    'essential_spending': 'gastos_esenciales',
    'income_type': 'tipo_ingreso',
    'rent_or_mortgage': 'alquiler_o_hipoteca',
    'category': 'categoria',
    'cash_flow_status': 'estado_flujo_caja',
    'financial_advice_score': 'puntaje_recomendacion_financiera',
    'financial_stress_level': 'nivel_estres_financiero',
    'actual_savings': 'ahorro_real',
    'savings_goal_met': 'meta_ahorro_cumplida'
}

df2.rename(columns=traduccion, inplace=True)
print(df2.columns.tolist())

['fecha', 'id_usuario', 'ingreso_mensual', 'gasto_mensual_total', 'tasa_ahorro', 'objetivo_presupuesto', 'escenario_financiero', 'puntaje_crediticio', 'relacion_deuda_ingreso', 'pago_prestamo', 'monto_inversion', 'servicios_suscripcion', 'fondo_emergencia', 'cantidad_transacciones', 'indicador_fraude', 'gastos_discrecionales', 'gastos_esenciales', 'tipo_ingreso', 'alquiler_o_hipoteca', 'categoria', 'estado_flujo_caja', 'puntaje_recomendacion_financiera', 'nivel_estres_financiero', 'ahorro_real', 'meta_ahorro_cumplida']


In [7]:
for columna in df2.columns:
    print(f"\n=== {columna} ===")
    print(df2[columna].unique())


=== fecha ===
<ArrowStringArray>
['2019-01-01', '2019-01-31', '2019-03-02', '2019-04-01', '2019-05-01',
 '2019-05-31', '2019-06-30', '2019-07-30', '2019-08-29', '2019-09-28',
 '2019-10-28', '2019-11-27', '2019-12-27', '2020-01-26', '2020-02-25',
 '2020-03-26', '2020-04-25', '2020-05-25', '2020-06-24', '2020-07-24',
 '2020-08-23', '2020-09-22', '2020-10-22', '2020-11-21', '2020-12-21',
 '2021-01-20', '2021-02-19', '2021-03-21', '2021-04-20', '2021-05-20',
 '2021-06-19', '2021-07-19', '2021-08-18', '2021-09-17', '2021-10-17',
 '2021-11-16', '2021-12-16', '2022-01-15', '2022-02-14', '2022-03-16',
 '2022-04-15', '2022-05-15', '2022-06-14', '2022-07-14', '2022-08-13',
 '2022-09-12', '2022-10-12', '2022-11-11', '2022-12-11', '2023-01-10',
 '2023-02-09', '2023-03-11', '2023-04-10', '2023-05-10', '2023-06-09',
 '2023-07-09', '2023-08-08', '2023-09-07', '2023-10-07', '2023-11-06']
Length: 60, dtype: str

=== id_usuario ===
[1584 1045 1756 1724 1600 1985 1194 1835 1454 1357 1403 1884 1675 1750


In [8]:
# Escenario financiero
df2["escenario_financiero"] = df2["escenario_financiero"].replace({
    "inflation": "Inflación",
    "normal": "Normal",
    "recession": "Recesión"
})

# Tipo de ingreso
df2["tipo_ingreso"] = df2["tipo_ingreso"].replace({
    "Freelance": "Independiente",
    "Salary": "Salario",
    "Mixed": "Mixto"
})

# Categoría
df2["categoria"] = df2["categoria"].replace({
    "Investments": "Inversiones",
    "Healthcare": "Salud",
    "Groceries": "Alimentacion",
    "Utilities": "Servicios",
    "Transportation": "Transporte",
    "Entertainment": "Ocio",
    "Education": "Educacion",
    "Insurance": "Otros",
    "Dining Out": "Gastos_hormiga",
    "Rent": "Vivienda"
})

# Estado del flujo de caja
df2["estado_flujo_caja"] = df2["estado_flujo_caja"].replace({
    "Positive": "Positivo",
    "Negative": "Negativo",
    "Neutral": "Neutral"
})

# Nivel de estrés financiero
df2["nivel_estres_financiero"] = df2["nivel_estres_financiero"].replace({
    "Low": "Bajo",
    "Medium": "Medio",
    "High": "Alto"
})
for columna in [
    "escenario_financiero",
    "tipo_ingreso",
    "categoria",
    "estado_flujo_caja",
    "nivel_estres_financiero"
]:
    print(f"\n{columna}:")
    print(df2[columna].unique())


escenario_financiero:
<ArrowStringArray>
['Inflación', 'Normal', 'Recesión']
Length: 3, dtype: str

tipo_ingreso:
<ArrowStringArray>
['Independiente', 'Salario', 'Mixto']
Length: 3, dtype: str

categoria:
<ArrowStringArray>
[   'Inversiones',          'Salud',   'Alimentacion',      'Servicios',
     'Transporte',           'Ocio',      'Educacion',          'Otros',
 'Gastos_hormiga',       'Vivienda']
Length: 10, dtype: str

estado_flujo_caja:
<ArrowStringArray>
['Positivo', 'Negativo', 'Neutral']
Length: 3, dtype: str

nivel_estres_financiero:
<ArrowStringArray>
['Bajo', 'Alto', 'Medio']
Length: 3, dtype: str


In [9]:
pd.set_option('display.max_columns', None)
df2

,fecha,id_usuario,ingreso_mensual,gasto_mensual_total,tasa_ahorro,objetivo_presupuesto,escenario_financiero,puntaje_crediticio,relacion_deuda_ingreso,pago_prestamo,monto_inversion,servicios_suscripcion,fondo_emergencia,cantidad_transacciones,indicador_fraude,gastos_discrecionales,gastos_esenciales,tipo_ingreso,alquiler_o_hipoteca,categoria,estado_flujo_caja,puntaje_recomendacion_financiera,nivel_estres_financiero,ahorro_real,meta_ahorro_cumplida
0,2019-01-01,1584,3119.58,3212.07,0.38,3676.11,Inflación,721.0,0.56,125.77,689.22,3,510.58,68,0,857.55,1910.85,Independiente,1501.65,Inversiones,Positivo,8.3,Bajo,0.00,0
1,2019-01-31,1045,3262.44,3732.81,0.10,2607.17,Inflación,670.0,0.42,454.19,360.34,4,1154.41,41,0,534.51,3165.20,Salario,1603.17,Inversiones,Positivo,22.6,Bajo,0.00,0
2,2019-03-02,1756,2931.20,3335.58,0.15,3004.14,Inflación,691.0,0.24,971.82,0.00,5,1433.02,90,0,353.67,1504.56,Independiente,1097.82,Salud,Positivo,58.8,Bajo,0.00,0
3,2019-04-01,1724,3506.79,2327.59,0.17,3346.97,Normal,717.0,0.16,482.76,182.06,5,227.37,94,0,594.08,1450.72,Independiente,1155.64,Alimentacion,Positivo,74.5,Bajo,1179.20,0
4,2019-05-01,1600,4606.87,2182.58,0.34,2670.09,Inflación,795.0,0.25,263.74,342.78,9,589.81,73,0,556.86,1000.00,Salario,1170.86,Servicios,Negativo,38.7,Alto,2424.29,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2995,2023-07-09,1822,3254.55,3992.02,0.17,2675.77,Normal,704.0,0.30,285.16,597.74,9,466.11,81,0,408.23,2088.32,Mixto,2079.99,Salud,Positivo,43.2,Bajo,0.00,0
2996,2023-08-08,1907,2752.78,5685.18,0.26,2682.99,Recesión,697.0,0.12,417.06,587.44,7,807.86,21,0,167.78,2445.34,Salario,832.95,Ocio,Neutral,46.0,Bajo,0.00,0
2997,2023-09-07,1464,3691.10,3228.05,0.12,3893.53,Normal,626.0,0.12,621.01,286.47,6,780.17,34,0,275.53,3231.56,Salario,790.68,Servicios,Neutral,40.7,Medio,463.05,0
2998,2023-10-07,1346,3133.24,4335.72,0.08,2854.72,Inflación,628.0,0.42,650.23,966.60,2,712.12,92,0,373.85,2305.14,Salario,903.82,Servicios,Positivo,57.1,Bajo,0.00,0


ETIQUETADO DEL PERFIL FINANCIERO

In [10]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler

# ==========================================================
# COPIA DEL DATASET
# ==========================================================

df = df2.copy()

# ==========================================================
# VARIABLES A NORMALIZAR
# ==========================================================

columnas_normalizar = [

    "ingreso_mensual",
    "gasto_mensual_total",
    "tasa_ahorro",
    "objetivo_presupuesto",
    "relacion_deuda_ingreso",
    "pago_prestamo",
    "monto_inversion",
    "servicios_suscripcion",
    "fondo_emergencia",
    "cantidad_transacciones",
    "gastos_discrecionales",
    "gastos_esenciales",
    "alquiler_o_hipoteca",
    "ahorro_real"

]

# ==========================================================
# NORMALIZACIÓN (SOLO COLUMNAS AUXILIARES)
# ==========================================================

scaler = MinMaxScaler()

df_norm = pd.DataFrame(

    scaler.fit_transform(df[columnas_normalizar]),

    columns=[

        columna + "_norm"

        for columna in columnas_normalizar

    ],

    index=df.index

)

df = pd.concat(

    [df, df_norm],

    axis=1

)

# ==========================================================
# VARIABLES CATEGÓRICAS AUXILIARES
# ==========================================================

mapa_flujo = {

    "Positivo": 1.0,
    "Neutral": 0.5,
    "Negativo": 0.0

}

mapa_estres = {

    "Bajo": 1.0,
    "Medio": 0.5,
    "Alto": 0.0

}

mapa_tipo = {

    "Mixto": 1.0,
    "Salario": 0.8,
    "Independiente": 0.6

}

df["estado_flujo_caja_score"] = (

    df["estado_flujo_caja"]

    .map(mapa_flujo)

)

df["nivel_estres_score"] = (

    df["nivel_estres_financiero"]

    .map(mapa_estres)

)

df["tipo_ingreso_score"] = (

    df["tipo_ingreso"]

    .map(mapa_tipo)

)

# ==========================================================
# ÍNDICE FINANCIERO
# ==========================================================

df["indice_financiero"] = (

      df["ingreso_mensual_norm"] * 10

    + (1 - df["gasto_mensual_total_norm"]) * 10

    + df["tasa_ahorro_norm"] * 18

    + (1 - df["relacion_deuda_ingreso_norm"]) * 18

    + (1 - df["pago_prestamo_norm"]) * 6

    + df["monto_inversion_norm"] * 8

    + (1 - df["servicios_suscripcion_norm"]) * 2

    + df["fondo_emergencia_norm"] * 8

    + (1 - df["gastos_discrecionales_norm"]) * 5

    + (1 - df["gastos_esenciales_norm"]) * 4

    + (1 - df["alquiler_o_hipoteca_norm"]) * 3

    + df["ahorro_real_norm"] * 4

    + df["estado_flujo_caja_score"] * 2

    + df["nivel_estres_score"] * 1

    + df["tipo_ingreso_score"] * 1

)

# ==========================================================
# AJUSTES DEL ÍNDICE
# ==========================================================

df["indice_financiero"] += (

    (

        1 -

        abs(

            df["objetivo_presupuesto_norm"]

            -

            df["ingreso_mensual_norm"]

        )

    ) * 3

)

df["indice_financiero"] += (

    df["meta_ahorro_cumplida"] * 2

)

df["indice_financiero"] = (

    df["indice_financiero"]

    .clip(0, 100)

)

# ==========================================================
# REVISIÓN DEL ÍNDICE
# ==========================================================

print("\nResumen del índice financiero\n")

print(

    df["indice_financiero"]

    .describe()

)

print("\nPrimeros registros\n")

print(

    df[[

        "ingreso_mensual",
        "gasto_mensual_total",
        "tasa_ahorro",
        "indice_financiero"

    ]].head()

)
# CREACIÓN DEL PERFIL FINANCIERO
# ==========================================================

# ----------------------------------------------------------
# CÁLCULO DE PERCENTILES
# ----------------------------------------------------------

p10 = df["indice_financiero"].quantile(0.10)
p30 = df["indice_financiero"].quantile(0.30)
p55 = df["indice_financiero"].quantile(0.55)
p75 = df["indice_financiero"].quantile(0.75)
p90 = df["indice_financiero"].quantile(0.90)

print("\nPercentiles")
print("--------------------------")
print(f"P10: {p10:.2f}")
print(f"P30: {p30:.2f}")
print(f"P55: {p55:.2f}")
print(f"P75: {p75:.2f}")
print(f"P90: {p90:.2f}")

# ----------------------------------------------------------
# FUNCIÓN DE ETIQUETADO
# ----------------------------------------------------------

def clasificar_perfil(indice):

    if indice <= p10:
        return "Crítico"

    elif indice <= p30:
        return "En riesgo"

    elif indice <= p55:
        return "En observación"

    elif indice <= p75:
        return "Estable"

    elif indice <= p90:
        return "Saludable"

    else:
        return "Excelente"

# ----------------------------------------------------------
# CREAR PERFIL FINANCIERO
# ----------------------------------------------------------

df["perfil_financiero"] = df["indice_financiero"].apply(
    clasificar_perfil
)

# ----------------------------------------------------------
# DISTRIBUCIÓN DE CLASES
# ----------------------------------------------------------

print("\nDistribución de perfiles")
print("--------------------------------")

print(df["perfil_financiero"].value_counts())

print("\nPorcentaje")

print(

    round(

        df["perfil_financiero"]

        .value_counts(normalize=True)

        *100,

        2

    )

)

# ----------------------------------------------------------
# ELIMINAR COLUMNAS AUXILIARES
# ----------------------------------------------------------

columnas_auxiliares = [

    columna + "_norm"

    for columna in columnas_normalizar

]

columnas_auxiliares += [

    "estado_flujo_caja_score",

    "nivel_estres_score",

    "tipo_ingreso_score"

]

df.drop(

    columns=columnas_auxiliares,

    inplace=True

)

# ----------------------------------------------------------
# REVISAR DATASET FINAL
# ----------------------------------------------------------

print("\nColumnas finales")

print(df.columns.tolist())

print("\nPrimeros registros")

print(df.head())

# ----------------------------------------------------------
# GUARDAR DATASET
# ----------------------------------------------------------

df.to_csv(

    "datasets/df_modelo_pf.csv",

    index=False

)

print("\n=========================================")
print("Dataset etiquetado guardado correctamente")
print("=========================================")


Resumen del índice financiero

count    3000.000000
mean       51.312420
std         8.539211
min        25.784765
25%        45.528431
50%        51.120958
75%        57.236037
max        77.725431
Name: indice_financiero, dtype: float64

Primeros registros

   ingreso_mensual  gasto_mensual_total  tasa_ahorro  indice_financiero
0          3119.58              3212.07         0.38          50.345998
1          3262.44              3732.81         0.10          39.294951
2          2931.20              3335.58         0.15          47.056776
3          3506.79              2327.59         0.17          53.087863
4          4606.87              2182.58         0.34          61.782912

Percentiles
--------------------------
P10: 40.23
P30: 46.63
P55: 52.35
P75: 57.24
P90: 62.46

Distribución de perfiles
--------------------------------
perfil_financiero
En observación    750
Estable           600
En riesgo         600
Saludable         450
Crítico           300
Excelente         300
Nam

# Transformando entrada de datos del usuario

Hacer un clasificador de ingresos de esta manera sabemos más acerca del perfil financiero de la persona

In [21]:
#Se recibe la tabla desde Backend
datos = {
    "fecha": [
        "2026-07-01",
        "2026-07-02",
        "2026-07-03",
        "2026-07-04",
        "2026-07-05",
        "2026-07-06",
        "2026-07-07",
        "2026-07-08",
        "2026-07-09",
        "2026-07-10",
        "2026-07-01"
    ],
    
    "descripcion": [
        "Sueldo mensual recibido",
        "Mercado Éxito",
        "Pago de energía",
        "Honorarios por consultoría",
        "Netflix",
        "Cobro de cliente",
        "Arriendo apartamento",
        "Gasolina",
        "Depósito de nómina",
        "Restaurante",
        "aporte a inversion"
    ],
    
    "categoria": [
        "Ingresos",
        "Alimentacion",
        "Servicios",
        "Ingresos",
        "Ocio",
        "Ingresos",
        "Vivienda",
        "Transporte",
        "Ingresos",
        "Gastos_hormiga",
        "Aporte_inversiones"
    ],
    
    "valor": [
        32,
        180,
        95,
        850,
        28,
        500,
        90,
        70,
        320,
        4.5,
        200
    ]
}

df_usuario = pd.DataFrame(datos)

print(df_usuario)

         fecha                 descripcion           categoria  valor
0   2026-07-01     Sueldo mensual recibido            Ingresos   32.0
1   2026-07-02               Mercado Éxito        Alimentacion  180.0
2   2026-07-03             Pago de energía           Servicios   95.0
3   2026-07-04  Honorarios por consultoría            Ingresos  850.0
4   2026-07-05                     Netflix                Ocio   28.0
5   2026-07-06            Cobro de cliente            Ingresos  500.0
6   2026-07-07        Arriendo apartamento            Vivienda   90.0
7   2026-07-08                    Gasolina          Transporte   70.0
8   2026-07-09          Depósito de nómina            Ingresos  320.0
9   2026-07-10                 Restaurante      Gastos_hormiga    4.5
10  2026-07-01          aporte a inversion  Aporte_inversiones  200.0


In [22]:
modelo_tipo_ingreso_cat = joblib.load("modelos/modelo_tipo_ingreso_1.pkl")
modelo_tipo_ingreso = joblib.load("modelos/modelo_tipo_ingreso_2.pkl")

In [23]:
#Analisis si hay o no ingresos
df_usuario["tipo_ingreso"] = None

# Filtrar solo las filas cuya categoría sea "Ingresos"
mask = df_usuario["categoria"] == "Ingresos"

# Si existen ingresos, clasificarlos
if mask.any():
    df_usuario.loc[mask, "tipo_ingreso_1"] = modelo_tipo_ingreso_cat.predict(
        df_usuario.loc[mask, "descripcion"]
    )

#Analisis si hay o no ingresos
df_usuario["tipo_ingreso"] = None

# Filtrar solo las filas cuya categoría no sea none
mask = df_usuario["tipo_ingreso_1"].notna()

# Si existen ingresos, clasificarlos
if mask.any():
    df_usuario.loc[mask, "tipo_ingreso"] = modelo_tipo_ingreso.predict(
        df_usuario.loc[mask, "tipo_ingreso_1"]
    )

# Para armar array que entra para perfil financiero tipo_ingreso

tipos = set(df_usuario["tipo_ingreso"].dropna())

if len(tipos) == 0:
    perfil_ingreso = "Sin ingresos registrados"
elif tipos == {"Salario"}:
    perfil_ingreso = "Salario"
elif tipos == {"Independiente"}:
    perfil_ingreso = "Independiente"
else:
    perfil_ingreso = "Mixto"

print(perfil_ingreso)

# Reemplazar la categoría con tipo_ingreso_1 cuando este no sea nulo
df_usuario["categoria"] = df_usuario["tipo_ingreso_1"].fillna(df_usuario["categoria"])

# Eliminar la columna tipo_ingreso_1
df_usuario.drop(columns=["tipo_ingreso_1"], inplace=True)

transacciones=df_usuario


Mixto


In [24]:
transacciones

,fecha,descripcion,categoria,valor,tipo_ingreso
0,2026-07-01,Sueldo mensual recibido,Salario,32.0,Salario
1,2026-07-02,Mercado Éxito,Alimentacion,180.0,None
2,2026-07-03,Pago de energía,Servicios,95.0,None
3,2026-07-04,Honorarios por consultoría,Honorarios,850.0,Independiente
4,2026-07-05,Netflix,Ocio,28.0,None
5,2026-07-06,Cobro de cliente,Negocio,500.0,Independiente
6,2026-07-07,Arriendo apartamento,Vivienda,90.0,None
7,2026-07-08,Gasolina,Transporte,70.0,None
8,2026-07-09,Depósito de nómina,Salario,320.0,Salario
9,2026-07-10,Restaurante,Gastos_hormiga,4.5,None


# Perfil Financiero

Transformar lo entrante para poderlo insertar en el modelo

In [ ]:
columnas_modelo = [
    "ingreso_mensual",
    "gasto_mensual_total",
    "tasa_ahorro",
    "objetivo_presupuesto",
    "relacion_deuda_ingreso",
    "pago_prestamo",
    "monto_inversion",
    "servicios_suscripcion",
    "fondo_emergencia",
    "cantidad_transacciones",
    "gastos_discrecionales",
    "gastos_esenciales",
    "tipo_ingreso",
    "alquiler_o_hipoteca",
    "estado_flujo_caja",
    "nivel_estres_financiero",
    "ahorro_real"
]
def generar_caracteristicas_usuario(
    ingreso_mensual,
    deuda_total,
    objetivo_presupuesto,
    pago_prestamo,
    servicios_suscripcion,
    fondo_emergencia,
    monto_inversion,
    transacciones
):

    categorias_ingreso = [
        "Honorarios",
        "Rendimiento_inversiones",
        "Negocio",
        "Otros_Ingresos",
        "Salario",
        "Subsidios"
    ]

    # Separar ingresos y gastos
    ingresos = transacciones[
        transacciones["categoria"].isin(categorias_ingreso)
    ]

    gastos = transacciones[
        ~transacciones["categoria"].isin(categorias_ingreso)
    ]


    # =========================
    # Cálculos financieros
    # =========================

    gasto_mensual_total = gastos["valor"].sum()


    gastos_esenciales = gastos.loc[
        gastos["categoria"].isin([
            "Alimentacion",
            "Educacion",
            "Salud",
            "Servicios",
            "Transporte",
            "Vivienda"
        ]),
        "valor"
    ].sum()


    gastos_discrecionales = gastos.loc[
        gastos["categoria"].isin([
            "Ocio",
            "Gastos_hormiga",
            "Otros"
        ]),
        "valor"
    ].sum()
    
    aporte_inversiones = gastos.loc[gastos["categoria"] == "Aporte_inversiones","valor"].sum()


    ahorro_real = ingreso_mensual - gasto_mensual_total + aporte_inversiones


    tasa_ahorro = (
        ahorro_real / ingreso_mensual
        if ingreso_mensual > 0 else 0
    )


    relacion_deuda_ingreso = (
        deuda_total / ingreso_mensual
        if ingreso_mensual > 0 else 0
    )
    
    
    # =========================
    # Inversiones
    # =========================

    monto_inversion = monto_inversion + aporte_inversiones

    # =========================
    # Vivienda
    # =========================

    alquiler_o_hipoteca = gastos.loc[
        gastos["categoria"] == "Vivienda",
        "valor"
    ].sum()



    # =========================
    # Flujo de caja
    # =========================

    if ahorro_real > 0:
        estado_flujo_caja = "Positivo"

    elif ahorro_real < 0:
        estado_flujo_caja = "Negativo"

    else:
        estado_flujo_caja = "Neutral"



    # =========================
    # Tipo de ingreso
    # =========================

    categorias_presentes = set(
        ingresos["categoria"]
    )


    tiene_salario = (
        "Salario" in categorias_presentes
    )


    tiene_independiente = len(
        categorias_presentes.intersection({
            "Honorarios",
            "Negocio",
            "Subsidios",
            "Rendimiento_inversiones",
            "Otros_Ingresos"
        })
    ) > 0



    if tiene_salario and tiene_independiente:
        tipo_ingreso = "Mixto"

    elif tiene_salario:
        tipo_ingreso = "Salario"

    elif tiene_independiente:
        tipo_ingreso = "Independiente"

    else:
        tipo_ingreso = "Sin ingreso"


    # =========================
    # Nivel de estrés financiero
    # Regla inicial
    # =========================

    porcentaje_deuda = relacion_deuda_ingreso

    if porcentaje_deuda > 0.5 or ahorro_real < 0:
        nivel_estres_financiero = "Alto"

    elif porcentaje_deuda > 0.3:
        nivel_estres_financiero = "Medio"

    else:
        nivel_estres_financiero = "Bajo"



    # =========================
    # Dataset final modelo
    # =========================

    datos_modelo = pd.DataFrame({

        "ingreso_mensual": [
            ingreso_mensual
        ],

        "gasto_mensual_total": [
            gasto_mensual_total
        ],

        "tasa_ahorro": [
            tasa_ahorro
        ],

        "objetivo_presupuesto": [
            objetivo_presupuesto
        ],

        "relacion_deuda_ingreso": [
            relacion_deuda_ingreso
        ],

        "pago_prestamo": [
            pago_prestamo
        ],

        "monto_inversion": [
            monto_inversion
        ],

        "servicios_suscripcion": [
            servicios_suscripcion
        ],

        "fondo_emergencia": [
            fondo_emergencia
        ],

        "cantidad_transacciones": [
            len(transacciones)
        ],

        "gastos_discrecionales": [
            gastos_discrecionales
        ],

        "gastos_esenciales": [
            gastos_esenciales
        ],

        "tipo_ingreso": [
            tipo_ingreso
        ],

        "alquiler_o_hipoteca": [
            alquiler_o_hipoteca
        ],

        "estado_flujo_caja": [
            estado_flujo_caja
        ],

        "nivel_estres_financiero": [
            nivel_estres_financiero
        ],

        "ahorro_real": [
            ahorro_real
        ]

    })

    return datos_modelo[columnas_modelo]

In [31]:
datos2  = generar_caracteristicas_usuario(
    ingreso_mensual=3000,
    deuda_total=1000,
    objetivo_presupuesto=500,
    pago_prestamo=200,
    servicios_suscripcion=2,
    fondo_emergencia=1500,
    monto_inversion=300,
    transacciones=transacciones
)

print(datos2)

   ingreso_mensual  gasto_mensual_total  tasa_ahorro  objetivo_presupuesto  \
0             3000                667.5     0.844167                   500   

   relacion_deuda_ingreso  pago_prestamo  monto_inversion  \
0                0.333333            200            500.0   

   servicios_suscripcion  fondo_emergencia  cantidad_transacciones  \
0                      2              1500                      11   

   gastos_discrecionales  gastos_esenciales tipo_ingreso  alquiler_o_hipoteca  \
0                   32.5              435.0        Mixto                 90.0   

  estado_flujo_caja nivel_estres_financiero  ahorro_real  
0          Positivo                   Medio       2532.5  


In [32]:
datos2.columns
datos2.dtypes
datos2.columns.tolist()

['ingreso_mensual',
 'gasto_mensual_total',
 'tasa_ahorro',
 'objetivo_presupuesto',
 'relacion_deuda_ingreso',
 'pago_prestamo',
 'monto_inversion',
 'servicios_suscripcion',
 'fondo_emergencia',
 'cantidad_transacciones',
 'gastos_discrecionales',
 'gastos_esenciales',
 'tipo_ingreso',
 'alquiler_o_hipoteca',
 'estado_flujo_caja',
 'nivel_estres_financiero',
 'ahorro_real']

In [33]:
modelo_pf = joblib.load("modelos/perfil_financiero.pkl")

## Tabla de Perfil Financiero
🟢 Excelente
🟢 Saludable 
🟡 Estable 
🟠 En observación 
🔴 En riesgo 
⚫ Crítico


In [34]:
perfil_financiero = modelo_pf.predict(datos2)

clases = modelo_pf.classes_

probabilidades = modelo_pf.predict_proba(datos2)[0]

resultado = dict(zip(clases, probabilidades))

print(resultado)
print(perfil_financiero)

{'Crítico': 0.013333333333333334, 'En observación': 0.05333333333333334, 'En riesgo': 0.02, 'Estable': 0.20666666666666667, 'Excelente': 0.39666666666666667, 'Saludable': 0.31}
['Excelente']
